**Setup:** download `data/zip/fintech.zip` from the DA Launchpad Datasets page and unzip it in the same folder as this notebook, so that paths like `fintech/<table>.csv` work. Needs pandas (and matplotlib/seaborn for charts).

# Where do UPI payments fail?

This notebook finds where and when payments fail. You will compare banks, apps and hours, find the peak pattern, and write three findings.

Put the CSV files for this dataset in a folder named `fintech` next to this notebook. The site zip has one folder per dataset. Run the cells from top to bottom.

In [ ]:
import pandas as pd

df = pd.read_csv("fintech/transactions.csv").drop_duplicates(subset="txn_id").copy()
df["ts"] = pd.to_datetime(df["txn_datetime"], format="%Y-%m-%d %H:%M:%S", errors="coerce").combine_first(
    pd.to_datetime(df["txn_datetime"], format="%d-%m-%Y %H:%M:%S", errors="coerce"))
df["hour"] = df["ts"].dt.hour
df["failed"] = df["status"].eq("FAILED")
print("transactions after dedupe:", len(df))

## Step 1: Overall status mix

First check the whole picture. The failure rate is the share of rows with `status` equal to `FAILED`.

In [ ]:
print((df["status"].value_counts(normalize=True) * 100).round(2))
print("overall failure rate %:", round(df["failed"].mean() * 100, 2))

## Step 2: Failure rate by bank

A rate is better than a count when banks have different volumes.

In [ ]:
by_bank = df.groupby("bank").agg(transactions=("txn_id", "size"), failure_rate_pct=("failed", "mean"))
by_bank["failure_rate_pct"] = (by_bank["failure_rate_pct"] * 100).round(2)
by_bank.sort_values("failure_rate_pct", ascending=False)

## Step 3: Failure rate by payment app

In [ ]:
by_app = df.groupby("payment_app").agg(transactions=("txn_id", "size"), failure_rate_pct=("failed", "mean"))
by_app["failure_rate_pct"] = (by_app["failure_rate_pct"] * 100).round(2)
by_app.sort_values("failure_rate_pct", ascending=False)

## Step 4: Failure rate by hour of the day

Look for a shape. Are there hours that stand out?

In [ ]:
by_hour = df.groupby("hour")["failed"].mean().mul(100).round(2)
by_hour.rename("failure_rate_pct").to_frame()

## Step 5: The peak pattern

Pivot the banks against the peak hours (9, 10, 18, 19 and 20). A bank that is high in many peak columns has a pattern, not bad luck.

In [ ]:
peak = df[df["hour"].isin([9, 10, 18, 19, 20])]
pivot = pd.pivot_table(peak, index="bank", columns="hour", values="failed", aggfunc="mean") * 100
pivot.round(1)

## Step 6: Failure reasons

The failed table has one row per failed attempt. Remove exact duplicates, then count the codes.

In [ ]:
fails = pd.read_csv("fintech/failed_transactions.csv").drop_duplicates()
print("failed attempts after dedupe:", len(fails))
fails["failure_code"].value_counts()

## Findings template

Write three findings. For each one, give the evidence (a number from a table above) and the action.

- Finding 1: the bank or app with the highest rate, and by how much.
- Finding 2: the hours with the highest rate, and whether they match the peak pattern.
- Finding 3: the most common failure code, and what it suggests (for example, a bank server issue or a balance issue).

Caveat: a failure rate does not show the cause. Check with the bank or the app team before you act.